<a href="https://www.kaggle.com/code/samratrm/kmeans-clustering-scratch-code?scriptVersionId=340570816" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [5]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import kagglehub

In [6]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# K-Means Clsutering Scratch code

In [37]:
class KMeans:
    def __init__(self, n_clusters=3, init="random", n_init=10, max_iter=300, tol=1e-4, random_state=None):
        self.n_clusters = n_clusters 
        self.init = init
        self.n_init = n_init
        self.max_iter = max_iter
        self.tol = tol 
        self.random_state = random_state

    @staticmethod
    def _to_array(X):
        if isinstance(X, (pd.DataFrame, pd.Series)):
            X = X.to_numpy()
        X = np.asarray(X, dtype=np.float64)
        
        if X.ndim != 2:
            raise ValueError("X must be 2D (N_samples, N_features")    
        return X 
    
    @staticmethod
    def _sq_distances(X, centroids):
        """Squared euclidean distances, shape (n_samples, k).
 
        Uses ||x - c||^2 = ||x||^2 - 2 x.c + ||c||^2 so the heavy part is a
        single matrix product (BLAS) instead of an (n, k, d) broadcast.
        """
        d = (
            (X**2).sum(axis=1)[:, None]
            - (2 * (X @ centroids.T)) 
            + (centroids ** 2).sum(axis=1)[None, :]
        )
        return np.maximum(d,0.0) # to chip off the tiny negative round-off

    def _init_centroids(self, X, rng):
        n = X.shape[0]
        if self.init.lower() == "random":
            idx = rng.choice(n, self.n_clusters, replace=False)
            return X[idx].copy()
        if self.init.lower() == "k-means++":
            centroids = np.empty((self.n_clusters, X.shape[1]))
            centroids[0] = X[rng.integers(n)]
            closest = self._sq_distances(X, centroids[:1]).ravel()
            for j in range(1, self.n_clusters):
                total = closest.sum()
                probs = np.full(n, 1.0/n) if total==0 else closest / total
                centroids[j] = X[rng.choice(n, p=probs)]
                closest = np.minimum(
                    closest, self._sq_distances(X, centroids[j:j+1]).ravel()
                )
            return centroids 
        else:
            raise ValueError("init must be 'K-means++' or 'random'")

    def _single_run(self, X, rng):
        # Assign centroids
        centroids = self._init_centroids(X, rng)
        labels = np.zeros(X.shape[0], dtype=np.int32)

        # update Centroids
        for it in range(1, self.max_iter+1):
            sq = self._sq_distances(X, centroids)
            labels = sq.argmin(axis=1)

            new_centroids = centroids.copy()
            for j in range(self.n_clusters):
                members = X[labels==j] 
                if len(members):
                    new_centroids[j] = members.mean(axis=0)
                else:
                    # empty cluster - move it to the worst fitted point
                    all_centroid_dist = sq[np.arange(len(X)), labels]
                    far_idx = all_centroid_dist.argmax()
                    new_centroids[j] = X[far_idx]

            shift = ((new_centroids - centroids) **2).sum()
            centroids = new_centroids
            if shift <= self.tol:
                break
        
        sq = self._sq_distances(X, centroids)
        labels = sq.argmin(axis=1)
        inertia = sq[np.arange(len(X)), labels].sum() # WCSS
        
        return centroids, labels, inertia, it 

    def fit(self, X, y=None):
        X = self._to_array(X)
        if not 1 <= self.n_clusters <= X.shape[0]:
            raise ValueError("need 1 <= n_clusters <= n_samples")
        rng = np.random.default_rng(self.random_state)

        best = None
        for _ in range(self.n_init):
            run = self._single_run(X, rng) # centroids, labels, WCSS, iteration
            if best is None or run[2] < best[2]:
                best = run 
        self.cluster_centers_, self.labels_, self.inertia_, self.n_iter_ = best 
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        X = self._to_array(X)
        if X.shape[1] != self.n_features_in_:
            raise ValueError("Feature count differnt from training data")
        return self._sq_distances(X, self.cluster_centers_).argmin(axis=1)
    
    def fit_predict(self, X, y=None):
        return self.fit(X).labels_

    def transform(self, X):
        X = self._to_array(X)
        return np.sqrt(self._sq.distances(X, self.cluster_centers_))

### Squared Distance Algebra Trick: 


For every point, its squared distance to every centroid. With 4 points and 2 centroids you get a 4×2 table:

|  | centroid 0 | centroid 1 |
|---|---|---|
| point 0 | ? | ? |
| point 1 | ? | ? |
| point 2 | ? | ? |
| point 3 | ? | ? |

"Squared" just means we skip the square root — `argmin` picks the same winner either way, so why pay for the `sqrt`.

**The algebra trick**

The obvious way to get the distance between point `x` and centroid `c` is subtract, square, sum:

```
(x - c)² summed over dimensions
```

But expand that by hand, like `(a-b)² = a² - 2ab + b²`:

```
‖x − c‖²  =  ‖x‖²  −  2·(x·c)  +  ‖c‖²
```

Same number, three separate pieces. That matters because `x·c` for *all* points against *all* centroids is exactly a matrix multiplication and NumPy hands matrix multiplication to BLAS, heavily optimized Fortran/C. The naive subtract-and-square approach instead builds a giant `n × k × d` array in memory first. Same answer, much slower and hungrier.

#### Worked example

```python
import numpy as np

X = np.array([[0., 0.],
              [1., 2.],
              [4., 4.],
              [5., 5.]])          # 4 points, 2 features

C = np.array([[0., 0.],
              [5., 5.]])          # 2 centroids
```

**Piece 1** — each point's squared length, `‖x‖²`:

```python
(X ** 2).sum(axis=1)      # → [ 0.,  5., 32., 50.]
```

`axis=1` means sum across the row. Point `[1,2]` gives `1+4 = 5`. Shape is `(4,)` — a flat list of 4.

`[:, None]` reshapes it to a **column**, shape `(4, 1)`:

```
[[ 0.]
 [ 5.]
 [32.]
 [50.]]
```

**Piece 3** — each centroid's squared length, `‖c‖²`:

```python
(C ** 2).sum(axis=1)      # → [ 0., 50.]
```

`[None, :]` makes it a **row**, shape `(1, 2)`:

```
[[ 0., 50.]]
```

**Piece 2** — the dot products, shape `(4, 2)`:

```python
X @ C.T
# [[  0.,   0.],
#  [  0.,  15.],
#  [  0.,  40.],
#  [  0.,  50.]]
```

Row 1 is point `[1,2]`: dotted with `[0,0]` gives 0, dotted with `[5,5]` gives `1·5 + 2·5 = 15`.

**Broadcasting glues them together**

Now the shapes are `(4,1)`, `(4,2)`, `(1,2)`. NumPy **stretches** the size-1 dimensions to match — the column repeats sideways, the row repeats downward:

```
   ‖x‖² column        −  2·(x·c)        +   ‖c‖² row
[[ 0,  0]              [[ 0,   0]         [[ 0, 50]
 [ 5,  5]        −    2·[ 0,  15]     +    [ 0, 50]
 [32, 32]               [ 0,  40]          [ 0, 50]
 [50, 50]]              [ 0,  50]]         [ 0, 50]]
```

Add it up, top-left entry: `0 − 0 + 0 = 0`. Point `[0,0]` sits exactly on centroid `[0,0]`. ✓

Second row, second column: `5 − 30 + 50 = 25`. Check by hand: `[1,2]` to `[5,5]` is `(1-5)² + (2-5)² = 16 + 9 = 25`. ✓

Full result:

```
[[  0., 50.],
 [  5., 25.],
 [ 32.,  2.],
 [ 50.,  0.]]
```

Read it as: point 0 belongs to centroid 0 (0 < 50), point 1 to centroid 0 (5 < 25), points 2 and 3 to centroid 1. That's the assignment step `d.argmin(axis=1)` → `[0, 0, 1, 1]`.

**Why `np.maximum(d, 0.0)`**

Distances can't be negative, but this formula can produce something like `-3e-16`. It's subtracting two nearly-equal big numbers — for a point sitting almost exactly on a centroid, `‖x‖² + ‖c‖²` and `2x·c` nearly cancel, and floating-point can't represent them precisely enough. The tiny negative is harmless for `argmin`, but `np.sqrt(-3e-16)` gives `nan`. So clamp anything below zero to zero.

`np.maximum(d, 0.0)` is elementwise: for each entry, keep whichever is larger, it or 0. (Not `np.max`, which would return one single largest number.)

**The two shorthands**

Worth memorizing, they show up constantly:

| Syntax | Effect |
|---|---|
| `arr[:, None]` | `(4,)` → `(4,1)`, a column |
| `arr[None, :]` | `(4,)` → `(1,4)`, a row |

`None` inserts a new axis of length 1. Getting one point-shape as a column and the other as a row is what makes broadcasting produce a full grid of every combination, rather than a single pairwise result.

In [38]:
def make_blobs(n=300, seed=0):
    rng = np.random.default_rng(seed)
    centres = np.array([[0.0, 0.0], [5.0, 5.0], [0.0, 6.0]])
    X = np.vstack([c + rng.normal(scale=0.9, size=(n // 3, 2)) for c in centres])
    return pd.DataFrame(X, columns=["x1", "x2"])

make_blobs().head()

,x1,x2
0,0.113157,-0.118894
1,0.576380,0.094410
2,-0.482102,0.325436
3,1.173600,0.852373
4,-0.633362,-1.138879


In [39]:
df = make_blobs()
 
km = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=42).fit(df)
def sort_rows(c):                      # sort centroids so runs are comparable
    return c[np.lexsort((c[:, 1], c[:, 0]))]

print("scratch centroids:\n", np.round(sort_rows(km.cluster_centers_), 3))
print(f"scratch inertia : {km.inertia_:.4f}   iters: {km.n_iter_}")

out = df.assign(cluster=km.labels_)
print("\ncluster means:\n", out.groupby("cluster").mean().round(3))
print("\nsizes:\n", out["cluster"].value_counts().sort_index().to_string())


scratch centroids:
 [[-0.064  0.091]
 [-0.053  6.062]
 [ 4.887  4.953]]
scratch inertia : 479.7674   iters: 4

cluster means:
             x1     x2
cluster              
0       -0.064  0.091
1        4.887  4.953
2       -0.053  6.062

sizes:
 cluster
0    100
1    100
2    100


In [43]:
from sklearn.cluster import KMeans as SkKMeans

sk = SkKMeans(n_clusters=3, n_init=10, random_state=42).fit(df)
print(f"\nsklearn inertia : {sk.inertia_:.4f}   iters: {sk.n_iter_}")
print("sklearn centroids:\n", np.round(sort_rows(sk.cluster_centers_), 3))
print(f"inertia gap     : {abs(km.inertia_ - sk.inertia_):.2e}")


sklearn inertia : 479.7674   iters: 5
sklearn centroids:
 [[-0.064  0.091]
 [-0.053  6.062]
 [ 4.887  4.953]]
inertia gap     : 3.41e-13
